# Member 2 — Feature Engineering Notebook
## CCS3356 Natural Language Processing | Fake News Detection
### Student ID: CIT-24-01-0486
### ML Feature: Bag of Words / TF-IDF (for Naïve Bayes)
### DL Feature: GloVe Word Embeddings (for LSTM)

In [1]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Imports done!")

Imports done!


In [2]:
df = pd.read_csv('../data/preprocessed.csv')
print("Data loaded!")
print("Shape:", df.shape)
df.head()

Data loaded!
Shape: (44898, 4)


,title,text,cleaned_text,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,"21st Century Wire says Ben Stein, reputable pr...",century wire say ben stein reputable professor...,1
1,Trump drops Steve Bannon from National Securit...,WASHINGTON (Reuters) - U.S. President Donald T...,washington reuters president donald trump remo...,0
2,Puerto Rico expects U.S. to lift Jones Act shi...,(Reuters) - Puerto Rico Governor Ricardo Rosse...,reuters puerto rico governor ricardo rossello ...,0
3,OOPS: Trump Just Accidentally Confirmed He Le...,"On Monday, Donald Trump once again embarrassed...",monday donald trump embarrassed country accide...,1
4,Donald Trump heads for Scotland to reopen a go...,"GLASGOW, Scotland (Reuters) - Most U.S. presid...",glasgow scotland reuters presidential candidat...,0


In [3]:
X = df['cleaned_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

Training samples: 35918
Testing samples:  8980

Train class distribution:
label
1    18785
0    17133
Name: count, dtype: int64

Test class distribution:
label
1    4696
0    4284
Name: count, dtype: int64


In [5]:
# Handle NaN values
X_train = X_train.fillna('')
X_test = X_test.fillna('')

print(f"NaN values in X_train: {X_train.isna().sum()}")
print(f"NaN values in X_test: {X_test.isna().sum()}")
print("NaN values handled!")

NaN values in X_train: 0
NaN values in X_test: 0
NaN values handled!


In [6]:
# CountVectorizer — Bag of Words
bow_vectorizer = CountVectorizer(max_features=50000, ngram_range=(1,2))
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

print("Bag of Words vectorizer fitted!")
print(f"Vocabulary size: {len(bow_vectorizer.vocabulary_)}")
print(f"Train matrix shape: {X_train_bow.shape}")
print(f"Test matrix shape:  {X_test_bow.shape}")

# Save vectorizer
with open('../models/m2_bow_vectorizer.pkl', 'wb') as f:
    pickle.dump(bow_vectorizer, f)
print("\nBoW vectorizer saved!")

Bag of Words vectorizer fitted!
Vocabulary size: 50000
Train matrix shape: (35918, 50000)
Test matrix shape:  (8980, 50000)

BoW vectorizer saved!


In [7]:
# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1,2))
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print("TF-IDF vectorizer fitted!")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")
print(f"Train matrix shape: {X_train_tfidf.shape}")
print(f"Test matrix shape:  {X_test_tfidf.shape}")

# Save vectorizer
with open('../models/m2_tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print("\nTF-IDF vectorizer saved!")

TF-IDF vectorizer fitted!
Vocabulary size: 50000
Train matrix shape: (35918, 50000)
Test matrix shape:  (8980, 50000)

TF-IDF vectorizer saved!


In [8]:
# Save splits for use in model notebooks
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)

print("Train/test splits saved!")
print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")

Train/test splits saved!
X_train: (35918,)
X_test:  (8980,)


## Feature Engineering Summary

### For Naïve Bayes (ML Model)
- **Bag of Words** — CountVectorizer with unigrams + bigrams, max 50,000 features
- **TF-IDF** — TfidfVectorizer with unigrams + bigrams, max 50,000 features
- Will compare both in the model notebook to pick the best

### For LSTM (DL Model)
- Keras Tokenizer fitted on training data only (max 50,000 words)
- Sequences padded to fixed length of 500 tokens
- GloVe 100d embeddings will be loaded in the model notebook

### Key Rule
- All vectorizers and tokenizers fitted on **training data only**
- Test data transformed using the fitted objects — no data leakage